# MLP: Baseline ohne Kaufgraph

In [1]:
import sys
import time
from pathlib import Path

import torch

sys.path.insert(0, "..")
from BuildFolds import build_folds
from KomplexDataLoader import BUY, ExplicitRelationalDataLoader, load_raw
from models.MLP import MLP
from Training import embedding_auc, generate_embeddings, model_name, record_results, train

## Einstellungen

In [2]:
GRAPH = "ausgelagert"   # oder "einfach": nur Kunden- und Produktknoten

DATA_DIR = "../../data"
CACHE_DIR = Path("cache/mlp" if GRAPH == "ausgelagert" else "cache/einfach/mlp")

CUSTOMER_FEATURES = ["yearOfBirth", "isMale", "premier"]
PRODUCT_FEATURES = ["avgGbpPrice", "avgDiscountValue"]
LABEL_FEATURES = ["sales", "returns", "rate", "hasHistory"]  # werden pro Fold neu berechnet
ARTICLE_NODE = GRAPH == "ausgelagert"
ARTICLE_FEATURES = GRAPH == "einfach"

K = 5
VAL_FRAC = 0.1
SPLIT_SEED = 42
LOG_COUNTS = True

MODEL_CLASS = MLP
TRAIN_FN = train
MODEL_PARAMS = dict(hidden_dim=64, out_dim=32)
NUM_NEIGHBORS = [10]
EVAL_NEIGHBORS = None
TRAIN_PARAMS = dict(epochs=30, lr=1e-3, batch_size=1024, num_neighbors=NUM_NEIGHBORS, patience=5, seed=0)

MODEL_NAME = model_name("MLP", GRAPH)

## 1. Rohdaten → heterogener Graph

In [3]:
durations = {}
t0 = time.perf_counter()
data = ExplicitRelationalDataLoader(
    *load_raw(DATA_DIR, "training"),
    customer_features=CUSTOMER_FEATURES,
    product_features=PRODUCT_FEATURES,
    label_features=LABEL_FEATURES,
    graph=GRAPH, article_node=ARTICLE_NODE, article_features=ARTICLE_FEATURES,
).data
print(data)

Achtung: 4954 product-Knoten ohne brand (es wird keine Kante angelegt)


Achtung: 71648 (5.23%) Events ohne src_cust-Knoten
Achtung: 474590 (34.66%) Events ohne dst_prod-Knoten


HeteroData(
  customer_features=[3],
  product_features=[2],
  label_features=[4],
  group_nodes={ article=[3] },
  customer={
    x=[777001, 3],
    reason=[777001, 13],
  },
  product={ x=[411495, 2] },
  product_type={ x=[11, 11] },
  country={ x=[9, 9] },
  brand={ x=[10, 10] },
  article={ x=[149161, 1] },
  (product, has_type, product_type)={ edge_index=[2, 411495] },
  (product_type, type_of, product)={ edge_index=[2, 411495] },
  (customer, lives_in, country)={ edge_index=[2, 777001] },
  (country, home_of, customer)={ edge_index=[2, 777001] },
  (product, has_brand, brand)={ edge_index=[2, 406541] },
  (brand, brand_of, product)={ edge_index=[2, 406541] },
  (product, variant_of, article)={ edge_index=[2, 411495] },
  (article, has_variant, product)={ edge_index=[2, 411495] },
  (customer, buys, product)={
    edge_index=[2, 848454],
    y=[848454],
    edge_id=[848454],
  },
  (product, bought_by, customer)={ edge_index=[2, 848454] }
)

## 2. Folds

In [4]:
folds, val = build_folds(data, k=K, val_frac=VAL_FRAC, seed=SPLIT_SEED, log_counts=LOG_COUNTS)
graphs = folds + ([val] if val is not None else [])

for g in graphs:
    assert not torch.isin(g[BUY].edge_label_id, g[BUY].edge_id).any()
    print(f"Fold {g.fold:2d}: {g[BUY].num_edges:7d} sichtbare Käufe im Graphen, "
          f"{g[BUY].edge_label.numel():7d} Zielkanten")

assert torch.equal(torch.cat([g[BUY].edge_label_id for g in graphs]).sort().values, data[BUY].edge_id)

reason = data["customer"].reason
del data
durations["Graph + Folds (Train)"] = time.perf_counter() - t0

Fold  0:  610887 sichtbare Käufe im Graphen,  152722 Zielkanten
Fold  1:  610887 sichtbare Käufe im Graphen,  152722 Zielkanten
Fold  2:  610887 sichtbare Käufe im Graphen,  152722 Zielkanten
Fold  3:  610887 sichtbare Käufe im Graphen,  152722 Zielkanten
Fold  4:  610888 sichtbare Käufe im Graphen,  152721 Zielkanten
Fold -1:  763609 sichtbare Käufe im Graphen,   84845 Zielkanten


## 3. Training

In [5]:
t0 = time.perf_counter()
torch.manual_seed(TRAIN_PARAMS["seed"])
model = MODEL_CLASS(folds[0].metadata(), **MODEL_PARAMS)
model, history = TRAIN_FN(model, folds, val, **TRAIN_PARAMS)
durations["Training"] = time.perf_counter() - t0

Epoche 01 | Loss 0.6136 | Val-AUC 0.7189


Epoche 02 | Loss 0.6088 | Val-AUC 0.7199


Epoche 03 | Loss 0.6082 | Val-AUC 0.7189


Epoche 04 | Loss 0.6077 | Val-AUC 0.7200


Epoche 05 | Loss 0.6074 | Val-AUC 0.7201


...

Epoche 15 | Loss 0.6059 | Val-AUC 0.7200


Epoche 16 | Loss 0.6057 | Val-AUC 0.7207


Epoche 17 | Loss 0.6057 | Val-AUC 0.7200


Epoche 18 | Loss 0.6056 | Val-AUC 0.7205


Early Stopping, beste Epoche 13 (Val-AUC 0.7209)


## 4. Embeddings

In [6]:
t0 = time.perf_counter()
embeddings = generate_embeddings(model, graphs, reason=reason, num_neighbors=EVAL_NEIGHBORS)
embeddings["params"] = dict(
    customer_features=CUSTOMER_FEATURES, product_features=PRODUCT_FEATURES, label_features=LABEL_FEATURES,
    graph=GRAPH, article_node=ARTICLE_NODE, article_features=ARTICLE_FEATURES, k=K, val_frac=VAL_FRAC, split_seed=SPLIT_SEED, log_counts=LOG_COUNTS,
    model=MODEL_CLASS.__module__, model_params=MODEL_PARAMS, train_params=TRAIN_PARAMS,
)

CACHE_DIR.mkdir(parents=True, exist_ok=True)
path = CACHE_DIR / "train_embeddings.pt"
torch.save(embeddings, path)
print(f"{len(embeddings['edge_id'])} Zustände, state {tuple(embeddings['state'].shape)} -> {path}")
durations["Embeddings (Train)"] = time.perf_counter() - t0

auc_values = dict(zip(["Train (OOF)", "Validierung"], embedding_auc(embeddings)))
params, state_dim = embeddings["params"], embeddings["state"].shape[1]
del embeddings, graphs, folds, val, g

848454 Zustände, state (848454, 64) -> cache/mlp/train_embeddings.pt


## 5. Testdaten: eigene Folds

In [7]:
t0 = time.perf_counter()
test_data = ExplicitRelationalDataLoader(
    *load_raw(DATA_DIR, "testing"),
    customer_features=CUSTOMER_FEATURES,
    product_features=PRODUCT_FEATURES,
    label_features=LABEL_FEATURES,
    graph=GRAPH, article_node=ARTICLE_NODE, article_features=ARTICLE_FEATURES,
).data
test_folds, _ = build_folds(test_data, k=K, val_frac=0, seed=SPLIT_SEED, log_counts=LOG_COUNTS)

for g in test_folds:
    assert not torch.isin(g[BUY].edge_label_id, g[BUY].edge_id).any()
    print(f"Test-Fold {g.fold}: {g[BUY].num_edges:7d} sichtbare Käufe im Graphen, "
          f"{g[BUY].edge_label.numel():7d} Zielkanten")

assert torch.equal(torch.cat([g[BUY].edge_label_id for g in test_folds]).sort().values, test_data[BUY].edge_id)

test_reason = test_data["customer"].reason
del test_data
durations["Graph + Folds (Test)"] = time.perf_counter() - t0

Achtung: 4962 product-Knoten ohne brand (es wird keine Kante angelegt)


Achtung: 75004 (5.14%) Events ohne src_cust-Knoten
Achtung: 451758 (30.93%) Events ohne dst_prod-Knoten


Test-Fold 0:  768615 sichtbare Käufe im Graphen,  192154 Zielkanten
Test-Fold 1:  768615 sichtbare Käufe im Graphen,  192154 Zielkanten
Test-Fold 2:  768615 sichtbare Käufe im Graphen,  192154 Zielkanten
Test-Fold 3:  768615 sichtbare Käufe im Graphen,  192154 Zielkanten
Test-Fold 4:  768616 sichtbare Käufe im Graphen,  192153 Zielkanten


## 6. Test-Embeddings

In [8]:
t0 = time.perf_counter()
test_embeddings = generate_embeddings(model, test_folds, reason=test_reason, num_neighbors=EVAL_NEIGHBORS)
test_embeddings["params"] = params

path = CACHE_DIR / "test_embeddings.pt"
torch.save(test_embeddings, path)
print(f"{len(test_embeddings['edge_id'])} Zustände, state {tuple(test_embeddings['state'].shape)} -> {path}")
durations["Embeddings (Test)"] = time.perf_counter() - t0

auc_values["Test"] = embedding_auc(test_embeddings)[0]

960769 Zustände, state (960769, 64) -> cache/mlp/test_embeddings.pt


## 7. Ergebnisse

In [9]:
record_results(MODEL_NAME, auc_values, durations, params, history=history, state_dim=state_dim)

MLP: AUC Train (OOF) 0.7211, AUC Validierung 0.7209, AUC Test 0.7212 | Dauer 2 min 05 s -> Ergebnisse.md
